In [3]:
import pandas as pd
import torch
from PIL import features
from torch.utils.data import Dataset, DataLoader, TensorDataset

In [4]:
delivery_df = pd.DataFrame({
    "trip_distance_miles": [2.5, 4.0, 6.0, 8.5, 10.0, 12.0, 15.0, 18.0],
    "pickup_hour": [8, 9, 17, 18, 7, 12, 16, 20],
    "passenger_count": [1, 1, 2, 2, 1, 3, 2, 1],
    "ride_duration_min": [9.0, 12.0, 18.0, 24.0, 27.0, 31.0, 39.0, 46.0]
})

In [5]:
delivery_df

,trip_distance_miles,pickup_hour,passenger_count,ride_duration_min
0,2.5,8,1,9.0
1,4.0,9,1,12.0
2,6.0,17,2,18.0
3,8.5,18,2,24.0
4,10.0,7,1,27.0
5,12.0,12,3,31.0
6,15.0,16,2,39.0
7,18.0,20,1,46.0


In [6]:
class DeliveryTransform:
    def __call__(self, feature):
        return torch.tensor(feature,dtype=torch.float32)

feature_transform = DeliveryTransform()
sample_features = delivery_df.loc[0, ["trip_distance_miles", "pickup_hour", "passenger_count"]].values
transform_feature = feature_transform(sample_features)

In [7]:
sample_features

array([2.5, 8. , 1. ])

In [8]:
transform_feature

tensor([2.5000, 8.0000, 1.0000])

In [9]:
transform_feature.shape

torch.Size([3])

In [10]:
class DeliveryDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform
        self.feature_col = ["trip_distance_miles", "pickup_hour", "passenger_count"]
        self.target_col = "ride_duration_min"

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.loc[index]
        features = row[self.feature_col].values
        target = row[self.target_col]

        if self.transform:
            features = self.transform(features)
        else:
            features = torch.tensor(features,dtype=torch.float32)

        target = torch.tensor([target], dtype=torch.float32)
        return features, target

delivery_dataset = DeliveryDataset(delivery_df, transform=feature_transform)

print("DataSet length:", len(delivery_dataset))
print("First sample:")
print(delivery_dataset[0])

DataSet length: 8
First sample:
(tensor([2.5000, 8.0000, 1.0000]), tensor([9.]))


In [12]:
first_feature, first_target = delivery_dataset[0]

print("First feature tensor:")
print(first_feature)
print("shape: ", first_feature.shape)
print("First target tensor:")
print(first_target)
print("shape: ", first_target.shape)

First feature tensor:
tensor([2.5000, 8.0000, 1.0000])
shape:  torch.Size([3])
First target tensor:
tensor([9.])
shape:  torch.Size([1])


In [13]:
delivery_loader = DataLoader(
    delivery_dataset,
    batch_size=3,
    shuffle=True,
)

In [14]:
print(delivery_loader)

In [19]:
for batch_feachers, batch_targets in delivery_loader:
    print("Batch feachers:")
    print(batch_feachers)
    print("shape: ", batch_feachers.shape)
    print("Batch targets:")
    print("shape: ", batch_targets.shape)
    print(batch_targets)
    break

Batch feachers:
tensor([[18., 20.,  1.],
        [ 6., 17.,  2.],
        [15., 16.,  2.]])
shape:  torch.Size([3, 3])
Batch targets:
shape:  torch.Size([3, 1])
tensor([[46.],
        [18.],
        [39.]])


In [20]:
import torch.nn.functional as F
from torchvision import datasets, transforms

In [21]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

print(mnist_transform)

Compose(
    ToTensor()
    Normalize(mean=(0.1307,), std=(0.3081,))
)
